In [2]:

# %pip install 'google-meridian'
# %pip install --upgrade pip


In [3]:
# ---- Set env vars BEFORE importing TensorFlow ----
import os

# # Hide most TensorFlow C++ INFO/WARNING logs
# # 0 = all logs, 1 = no INFO, 2 = no INFO/WARN, 3 = only fatal
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # adjust if you want more/less logs [web:21][web:23]

# # Optional: disable oneDNN custom ops if you don't want that message
# # or if you care about strict numerical reproducibility across devices.
# os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # comment this out if you want oneDNN optimizations [web:7][web:29]


# ---- Your existing imports ----
import arviz as az
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.data import test_utils
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.analysis import formatter  # this is the documented location
from importlib.metadata import version, PackageNotFoundError 
import numpy as np
import pandas as pd

import pickle

# NumPy 1.x compatibility shim for Meridian 1.3.2
if not hasattr(np, 'concat'):
    np.concat = np.concatenate
    print("✅ Added np.concat compatibility shim")


from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

# ---- Resource checks ----
ram_gb = virtual_memory().total / 1e9
print("Your runtime has {:.1f} gigabytes of available RAM\n".format(ram_gb))

cpus = tf.config.list_physical_devices("CPU")
gpus = tf.config.list_physical_devices("GPU")  # modern API (no experimental) [web:26][web:28]

print("Num CPUs Available: ", len(cpus))
print("Num GPUs Available: ", len(gpus))

if gpus:
    print("GPU devices:")
    for gpu in gpus:
        print("  -", gpu)
else:
    print("No GPU detected by TensorFlow")

# ---- Simple GPU test (optional) ----
if gpus:
    try:
        with tf.device("/GPU:0"):
            a = tf.random.uniform((1000, 1000))
            b = tf.random.uniform((1000, 1000))
            c = tf.matmul(a, b)
        print("Simple matmul on GPU succeeded. Tensor shape:", c.shape)
    except Exception as e:
        print("GPU test failed, falling back to CPU. Error:", e)


print("✅ Libraries imported")
import warnings
warnings.filterwarnings('ignore')

print("✅ Meridian libraries imported")

Your runtime has 8.6 gigabytes of available RAM

Num CPUs Available:  1
Num GPUs Available:  0
No GPU detected by TensorFlow
✅ Libraries imported
✅ Meridian libraries imported


In [4]:
df = pd.read_csv("final_ds.csv")
df.head()

,Unnamed: 0,geo,time,TV_impression,Meta_Ads_impression,TikTok_impression,Google_Search_impression,YouTube_impression,competitor_sales_control,sentiment_score_control,...,revenue_per_conversion,population,year,month,week,day_of_week,quarter,total_spend,digital_spend,traditional_spend
0,0,Geo0,2021-01-25,280668,0,0,470611,108010,-1.338765,0.115581,...,0.020055,136670.94,2021,1,4,0,1,6567.06170,4509.00090,2058.0608
1,1,Geo0,2021-02-01,366206,182108,19825,527702,252506,0.893645,0.944224,...,0.020103,136670.94,2021,2,5,0,1,10668.15268,7982.86528,2685.2874
2,2,Geo0,2021-02-08,197565,230170,0,393618,184061,-0.284549,-1.290579,...,0.019929,136670.94,2021,2,6,0,1,8169.40110,6720.71160,1448.6895
3,3,Geo0,2021-02-15,140990,66643,0,326034,201729,-1.034740,-1.084514,...,0.019987,136670.94,2021,2,7,0,1,5788.94667,4755.10607,1033.8406
4,4,Geo0,2021-02-22,399116,164991,0,381982,153973,-0.319276,-0.017503,...,0.020000,136670.94,2021,2,8,0,1,8693.79250,5767.18530,2926.6072


In [5]:
# Initialize DataFrameInputDataBuilder
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue',
    default_kpi_column='conversions',
    default_revenue_per_kpi_column='revenue_per_conversion',
)

print("✅ Builder initialized")

✅ Builder initialized


In [6]:
builder = (
    builder.with_kpi(df)
    .with_revenue_per_kpi(df)
    .with_population(df)
    .with_controls(
        df, control_cols=["sentiment_score_control", "competitor_sales_control"]
    )
)

# Define media channels (using renamed channels)
channels = ["TV", "Meta_Ads", "TikTok", "Google_Search", "YouTube"]

# Add media data
builder = builder.with_media(
    df,
    media_cols=[f"{channel}_impression" for channel in channels],
    media_spend_cols=[f"{channel}_spend" for channel in channels],
    media_channels=channels,
)


In [7]:
builder = builder.with_non_media_treatments(
    df, non_media_treatment_cols=['Promo']
).with_organic_media(
    df,
    organic_media_cols=['Organic_TV_impression'],
    organic_media_channels=['Organic_TV'],
)

In [8]:
# Build the InputData object
data = builder.build()
print("✅ InputData built successfully")

✅ InputData built successfully


In [9]:
# Set ROI priors (from official documentation)
roi_mu = 0.2      # Mu for ROI prior
roi_sigma = 0.9   # Sigma for ROI prior

# Create prior distribution
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)

# Create model specification
model_spec = spec.ModelSpec(prior=prior, enable_aks=True)

mmm = model.Meridian(input_data=data, model_spec=model_spec)

print("✅ Model specification created")
print(f"   ROI Prior: LogNormal(μ={roi_mu}, σ={roi_sigma})")

2025-12-11 21:54:41.486108: I external/local_xla/xla/service/service.cc:163] XLA service 0x6000017b0500 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2025-12-11 21:54:41.486280: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version


✅ Model specification created
   ROI Prior: LogNormal(μ=0.2, σ=0.9)


I0000 00:00:1765470281.734176  276696 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [ ]:
%%time
mmm.sample_prior(200)
mmm.sample_posterior(
    n_chains=2, n_adapt=400, n_burnin=200, n_keep=300, seed=0
)


2025-12-11 21:57:42.614397: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-11 21:57:48.814751: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator mcmc_retry_init/assert_equal_1/Assert/AssertGuard/Assert


In [16]:
import pickle

#BASE_PATH = "/Workspace/COMM - Commercial Analytics (CMAN)/MMM Quattro 2025/Satish/application_demo"

#model_path = f"{BASE_PATH}/meridian_model.pkl"

# Save model
with open("meridian_model.pkl".replace('file:', ''), 'wb') as f:
    pickle.dump(mmm, f)

print("=" * 80)
print("💾 MODEL SAVED")
print("=" * 80)
#print(f"📁 Location: {model_path}")
print(f"📊 Channels: {len(channels)}")
print(f"📈 Samples: 5 chains × 1000 samples = 5,000 total")

💾 MODEL SAVED
📊 Channels: 5
📈 Samples: 5 chains × 1000 samples = 5,000 total


In [17]:
print("=" * 80)
print("✅ NOTEBOOK 3 COMPLETE")
print("=" * 80)
print("🎉 Model training successful!")

✅ NOTEBOOK 3 COMPLETE
🎉 Model training successful!
